In [2]:
import pandas as pd
import numpy as np
import pandas_ta as ta

In [3]:
df = pd.read_csv('btc_data_clean.csv', parse_dates=['Open time'])
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,ignore
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999,5.993910e+06,5228,228.521921,3.090541e+06,0
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999,5.154522e+06,4534,180.840403,2.430449e+06,0
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,2018-01-01 02:59:59.999,5.710192e+06,4887,192.237935,2.558505e+06,0
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,2018-01-01 03:59:59.999,5.657448e+06,4789,137.918407,1.858041e+06,0
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,2018-01-01 04:59:59.999,4.588047e+06,4563,172.957635,2.328058e+06,0


In [6]:
# RSI 14
df['rsi_14'] = ta.rsi(df['close'], length=14)

In [7]:
# RSI Divergence
df['price_high'] = df['close'].rolling(14).max()
df['price_low']  = df['close'].rolling(14).min()
df['rsi_high']   = df['rsi_14'].rolling(14).max()
df['rsi_low']    = df['rsi_14'].rolling(14).min()

In [8]:
bearish_div = (df['close'] == df['price_high']) & (df['rsi_14'] < df['rsi_high'])
bullish_div = (df['close'] == df['price_low'])  & (df['rsi_14'] > df['rsi_low'])

df['rsi_divergence'] = 0
df.loc[bullish_div, 'rsi_divergence'] = 1
df.loc[bearish_div, 'rsi_divergence'] = -1

df.drop(columns=['price_high', 'price_low', 'rsi_high', 'rsi_low'], inplace=True)

df.columns

Index(['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time',
       'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume',
       'taker_buy_quote_asset_volume', 'ignore', 'rsi_14', 'rsi_divergence'],
      dtype='str')

In [9]:
# OBV Change (10 bars)
direction = df['close'].diff(1).apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
obv = (df['volume'] * direction).cumsum()
df['obv_change'] = obv.diff(10)

In [10]:
# Volume ROC (10 bars)
df['volume_roc'] = df['volume'].pct_change(periods=10)

In [11]:
# Summary
feature_cols = ['rsi_14', 'rsi_divergence', 'obv_change', 'volume_roc']
print(f"New columns: {feature_cols}\n")
print(df[feature_cols].describe().round(4))
print(f"\nRSI Divergence distribution:\n{df['rsi_divergence'].value_counts().sort_index()}")
print(f"\nNull counts:\n{df[feature_cols].isnull().sum()}")

New columns: ['rsi_14', 'rsi_divergence', 'obv_change', 'volume_roc']

           rsi_14  rsi_divergence   obv_change  volume_roc
count  71587.0000      71588.0000   71578.0000  71578.0000
mean      50.6276         -0.0040     110.7614      0.4021
std       12.2461          0.2456   15105.9519      2.1689
min        0.0000         -1.0000 -296382.9787     -0.9973
25%       42.8425          0.0000   -3502.3650     -0.3995
50%       50.6447          0.0000      16.5478     -0.0110
75%       58.3234          0.0000    3427.7425      0.6452
max       96.4105          1.0000  260507.6221    349.7934

RSI Divergence distribution:
rsi_divergence
-1     2303
 0    67270
 1     2015
Name: count, dtype: int64

Null counts:
rsi_14             1
rsi_divergence     0
obv_change        10
volume_roc        10
dtype: int64
